# Discord Persona Inference: Mistral NeMo 12B

This notebook is dedicated exclusively to running multi-turn chat inference with your fine-tuned LoRA adapter. It is designed to be lightweight and state-aware, meaning it will skip redundant downloads if you need to restart the Colab runtime.

### Features:
* **Multi-Turn Memory:** The clone remembers the conversation history for realistic context.
* **Encrypted Adapter Support:** Automatically downloads and decrypts your adapter from Google Drive.
* **Disk Persistence:** Leverages Hugging Face caching and local directory checks to bypass 20 minute wait times on session restarts.
* **4-bit Quantization:** Fits the 12B base model and your adapter into a free T4 GPU.

In [ ]:
# ==========================================
# USER CONFIGURATION PARAMETERS
# ==========================================

# 1. Adapter (Clone) Source Configuration
# The Google Drive share link to your encrypted 'final_adapter_encrypted.zip'
GDRIVE_ADAPTER_LINK = ''
# The AES-256 password used to encrypt the adapter during training
DECRYPTION_KEY = ""

# 2. Base Model Configuration
# Hugging Face token required to download Mistral-NeMo-Instruct-2407
HF_TOKEN = ""

# 3. Inference Hyperparameters
SYSTEM_PROMPT = "You are my Discord clone. Chat casually."
MAX_NEW_TOKENS = 150
TEMPERATURE = 0.7
TOP_P = 0.9

In [ ]:
import os
import time
import datetime
from google.colab import drive

drive.mount('/content/drive')

def log_print(message):
    current_time = datetime.datetime.now().strftime("%H:%M:%S")
    print(f"[{current_time}] {message}")

log_print("Environment initialized.")

In [ ]:
log_print("Installing dependencies...")
!pip install -q transformers peft bitsandbytes gdown accelerate pyzipper huggingface_hub
log_print("Dependencies installed.")

In [ ]:
import gdown
import pyzipper
import shutil

local_adapter_zip = '/content/adapter.zip'
local_adapter_dir = '/content/adapter_local'

def get_gdrive_id(url):
    if '/d/' in url:
        return url.split('/d/')[1].split('/')[0]
    elif 'id=' in url:
        return url.split('id=')[1].split('&')[0]
    return url

log_print("Checking for existing adapter...")
if os.path.exists(local_adapter_dir) and os.path.isdir(local_adapter_dir) and len(os.listdir(local_adapter_dir)) > 0:
    log_print(f"Adapter already extracted at: {local_adapter_dir}. Skipping download.")
else:
    if GDRIVE_ADAPTER_LINK:
        log_print("Downloading adapter zip from Google Drive via gdown...")
        file_id = get_gdrive_id(GDRIVE_ADAPTER_LINK)
        download_url = f'https://drive.google.com/uc?id={file_id}'
        gdown.download(download_url, local_adapter_zip, quiet=False)
        
        if os.path.exists(local_adapter_zip):
            os.makedirs(local_adapter_dir, exist_ok=True)
            log_print("Extracting adapter...")
            try:
                if DECRYPTION_KEY:
                    log_print("Using DECRYPTION_KEY for AES-256 decryption...")
                    with pyzipper.AESZipFile(local_adapter_zip, 'r', compression=pyzipper.ZIP_DEFLATED, encryption=pyzipper.WZ_AES) as zip_ref:
                        zip_ref.pwd = DECRYPTION_KEY.encode('utf-8')
                        zip_ref.extractall(local_adapter_dir)
                else:
                    with pyzipper.AESZipFile(local_adapter_zip, 'r') as zip_ref:
                        zip_ref.extractall(local_adapter_dir)
                log_print("Adapter successfully extracted.")
                os.remove(local_adapter_zip)
            except Exception as e:
                log_print(f"Extraction failed (Check password): {e}")
    else:
        log_print("WARNING: No GDRIVE_ADAPTER_LINK provided.")

In [ ]:
import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from huggingface_hub import login

log_print("Executing aggressive memory clearing...")
for var_name in ['model', 'base_model', 'tokenizer']:
    if var_name in globals():
        del globals()[var_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

if HF_TOKEN:
    log_print("Authenticating with Hugging Face...")
    login(HF_TOKEN)
else:
    log_print("Warning: No HF_TOKEN provided. Download will likely fail.")

base_model_id = "mistralai/Mistral-Nemo-Instruct-2407"

log_print("Configuring 4-bit quantization...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

log_print("Loading Tokenizer (Slow mode)... ")
tokenizer = AutoTokenizer.from_pretrained(base_model_id, use_fast=False)
tokenizer.pad_token = tokenizer.eos_token

log_print("Loading Base Model (Will use cached files if runtime was merely restarted)...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.float16
)
base_model.config.torch_dtype = torch.float16
base_model.config.use_cache = True

log_print("Merging LoRA Adapter...")
try:
    model = PeftModel.from_pretrained(base_model, local_adapter_dir)
    log_print("Model successfully loaded and merged.")
except Exception as e:
    log_print(f"Failed to merge adapter. Ensure the adapter directory contains the weights: {e}")

In [ ]:
# ******************************************
# Multi-Turn Chat Interface
# ******************************************
conversation_history = [
    {"role": "system", "content": SYSTEM_PROMPT}
]

def chat_with_clone(user_message):
    conversation_history.append({"role": "user", "content": user_message})
    
    prompt = tokenizer.apply_chat_template(conversation_history, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    outputs = model.generate(
        **inputs, 
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    
    conversation_history.append({"role": "assistant", "content": response})
    
    return response

print("\n" + "*"*50)
print("Clone is ready! Type 'quit' to exit.")
print("Type 'reset' to clear the conversation memory.")
print("*"*50 + "\n")

while True:
    user_input = input("You: ")
    if user_input.lower() in ['quit', 'exit', 'stop']:
        print("Ending chat.")
        break
    elif user_input.lower() == 'reset':
        conversation_history = [{"role": "system", "content": SYSTEM_PROMPT}]
        print("[Memory cleared.]\n")
        continue
    
    reply = chat_with_clone(user_input)
    print(f"Clone: {reply}\n")